# Register a trained model, walk it through the Model Registry approval workflow, and deploy the approved version to a Serverless Inference endpoint

A platform team has been deploying SageMaker endpoints by passing a `model.tar.gz` path directly to `CreateEndpoint`. The audit lead just blocked the practice because there is no versioned record of which model is serving traffic, no documented approval path, and no rollback story. You have one session to retrofit a versioned approval workflow on top of the platform's current pattern, deploy from the registry, then register a second version, reject it, and prove the rejected version never reaches production traffic.

Five artifacts ship in this lab:

- A SageMaker Model Package Group named `labrun-model-registry-and-approval-workflow-group` that acts as the per-model lineage container.
- Two XGBoost training jobs against a small deterministic 2000-row tabular set, producing two distinct `model.tar.gz` artifacts.
- A Model Package Version 1 registered with `ModelApprovalStatus=PendingManualApproval`, transitioned to `Approved` via `update_model_package`, deployed to a Serverless Inference endpoint, and serving 10 test predictions.
- A Model Package Version 2 registered with `PendingManualApproval` and immediately transitioned to `Rejected`. No deployment.
- A checkpoint that proves the deployed endpoint's underlying model artifact matches v1 and NOT v2 (the audit control actually controls).

**Time.** Plan for about 55 minutes hands-on. Each XGBoost training job runs roughly 5 minutes on ml.m5.large; the Serverless endpoint deploys in under 90 seconds; the registry calls return in milliseconds. Budget 90 minutes if the inference container URI fights back.

**Cost.** This lab costs about two cents if both training jobs land on the first try. SageMaker training on ml.m5.large is $0.115 per hour; two 5-minute runs is roughly $0.02. The Serverless Inference endpoint runs 10 invocations in this lab, which costs fractions of a penny ($0.20 per 1M requests plus per-GB-second). Model Registry itself is free. S3 storage for two training sets and two model artifacts is fractions of a penny. A realistic session with one registry retry lands around $0.02 to $0.05. The cleanup cell shuts the endpoint, the model, the package versions, the group, and the bucket down so your bill stops the moment you finish.

This lab maps to AWS MLA-C01 Domain 2 (ML Model Development, 26%) on versioning models via SageMaker Model Registry with the ModelApprovalStatus workflow, and to Domain 3 (Deployment and Orchestration of ML Workflows, 22%) on choosing deployment infrastructure (Serverless Inference) and building CI/CD with the approval pattern.

**The audit story.** "Pass a model.tar.gz path straight to CreateEndpoint" leaves no record of which model is serving traffic, who approved it, or whether it was the same artifact the team trained yesterday or last week. Model Registry replaces that with a Model Package Group (per-model lineage), Model Package Versions (per-deployment history), and ModelApprovalStatus transitions (approval audit trail). When security asks "which model is serving traffic and who approved it," you have an API answer.


In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the SageMaker Python SDK for image URI lookups.
# Both are pinned to specific versions per LAB-CREATION-BLUEPRINT.md.

!pip install --quiet labrun-checks==0.6.0 sagemaker==2.232.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import csv
import getpass
import io
import json
import sys
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "model-registry-and-approval-workflow"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

# Resource names. Bucket name is account-suffixed and resolved after STS returns.
TRAINING_ROLE_NAME = f"labrun-{LAB_ID}-role"
INLINE_POLICY_NAME = "labrun-mla-lab07-role-policy"
TRAINING_JOB_V1 = f"labrun-{LAB_ID}-job-v1"
TRAINING_JOB_V2 = f"labrun-{LAB_ID}-job-v2"
GROUP_NAME = f"labrun-{LAB_ID}-group"
ENDPOINT_CONFIG_NAME = f"labrun-{LAB_ID}-endpoint-config"
ENDPOINT_NAME = f"labrun-{LAB_ID}-endpoint"
MODEL_NAME = f"labrun-{LAB_ID}-model"

BUCKET_NAME = None
ACCOUNT_ID = None
TRAINING_ROLE_ARN = None
XGBOOST_IMAGE_URI = None
PACKAGE_V1_ARN = None
PACKAGE_V2_ARN = None
TRAINING_V1_ARTIFACT = None
TRAINING_V2_ARTIFACT = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# resolve account-derived names. This cell must succeed before the manifest
# cell creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"MLA Batch 2 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
TRAINING_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{TRAINING_ROLE_NAME}"
print(f"Bucket: {BUCKET_NAME}")
print(f"Training role ARN: {TRAINING_ROLE_ARN}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan.
# Manifest is module-level and in reverse-creation order.
#
# labrun-checks 0.6.0 does not have adapters for sagemaker_endpoint,
# sagemaker_endpoint_config, sagemaker_model, sagemaker_model_package, or
# sagemaker_model_package_group. The cleanup cell tears those down manually
# BEFORE run_cleanup walks the manifest. The manifest below contains only
# adapter-supported types (iam_role, s3_bucket). The orphan scan still picks
# up any tagged SageMaker resource via Resource Groups Tagging API.
#
# The Serverless endpoint is the critical resource because it is the only
# resource here that can survive a kernel restart with a billing surface
# (Serverless is $0 idle but the endpoint config and model still need to
# come down, and a stale endpoint name blocks a future re-run).

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iam_role",
        id=TRAINING_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {TRAINING_ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Run the cleanup")
    print("cell at the bottom of this notebook first, or remove them manually with")
    print("the AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Upload training and validation CSV, run two XGBoost training jobs with different hyperparameters

The first leg of the registry workflow is two distinct model artifacts to register as two distinct versions. XGBoost built-in algorithm expects CSV without a header row and the label in the first column. Generate a deterministic 2000-row binary-classification set, write the 70/30 train/validation split to S3, then submit two training jobs that differ only in `max_depth` and `eta`.

Build it in this order:

1. Create the S3 bucket `labrun-model-registry-and-approval-workflow-{your-account-id}` and tag it `labrun:lab-slug=model-registry-and-approval-workflow`.
2. Generate the 2000-row CSV (12 numeric features, binary label, deterministic via `random.seed(42)`), split 70/30 into `input/train.csv` and `input/validation.csv`, upload both. Label is in column 0; no header row.
3. Create the IAM role `labrun-model-registry-and-approval-workflow-role` with `sagemaker.amazonaws.com` trust and an inline policy granting S3 read/write on the bucket plus `sagemaker:InvokeEndpoint`.
4. Submit two `sm.create_training_job` calls. Both use the XGBoost built-in container URI for `us-east-1`. Job v1 uses `max_depth=5, eta=0.2`. Job v2 uses `max_depth=8, eta=0.1`. Both run on ml.m5.large for roughly 5 minutes each.
5. Poll until both reach `Completed`.

The two artifacts are what Tasks 2-5 register, approve, deploy, and reject.

In [ ]:
# NBVAL_SKIP
# Task 1: Create bucket, role, training data, and submit two training jobs.

import random
import sagemaker

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sm = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the bucket with s3.create_bucket(Bucket=BUCKET_NAME).
# us-east-1 rejects CreateBucketConfiguration; other regions require it.

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")


def generate_xgboost_csv(n_rows: int) -> bytes:
    rng = random.Random(42)
    buf = io.StringIO()
    writer = csv.writer(buf)
    for _ in range(n_rows):
        features = [round(rng.gauss(0, 1), 4) for _ in range(12)]
        # Label is a logistic of the first three features with noise.
        logit = 0.6 * features[0] - 0.4 * features[1] + 0.3 * features[2] + rng.gauss(0, 0.3)
        label = 1 if logit > 0 else 0
        writer.writerow([label] + features)
    return buf.getvalue().encode("utf-8")


full = generate_xgboost_csv(2000)
split_at = int(len(full.splitlines()) * 0.7)
lines_all = full.splitlines(keepends=True)
train_bytes = b"".join(lines_all[:split_at])
val_bytes = b"".join(lines_all[split_at:])

# YOUR CODE: Upload train_bytes to s3://{BUCKET_NAME}/input/train.csv and
# val_bytes to s3://{BUCKET_NAME}/input/validation.csv using s3.put_object().
print(f"Uploaded s3://{BUCKET_NAME}/input/train.csv ({len(train_bytes)} bytes)")
print(f"Uploaded s3://{BUCKET_NAME}/input/validation.csv ({len(val_bytes)} bytes)")

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    # YOUR CODE: Create the role using iam.create_role() with
    # RoleName=TRAINING_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy),
    # Description="labrun mla lab 07 training and endpoint role", and
    # Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
    print(f"Created role: {TRAINING_ROLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Role {TRAINING_ROLE_NAME} already exists, reusing.")
    else:
        raise

inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BucketRW",
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*",
        },
        {
            "Sid": "BucketList",
            "Effect": "Allow",
            "Action": "s3:ListBucket",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
        },
    ],
}

# YOUR CODE: Attach the inline policy using iam.put_role_policy() with
# RoleName=TRAINING_ROLE_NAME, PolicyName=INLINE_POLICY_NAME,
# PolicyDocument=json.dumps(inline_policy).
print(f"Attached inline policy {INLINE_POLICY_NAME} to {TRAINING_ROLE_NAME}")

# IAM propagation. create_training_job rejects roles that have not finished
# propagating; give it 10 seconds.
print("Your IAM role is stuck in traffic, give it 10 seconds...")
time.sleep(10)

# Fetch the XGBoost built-in image URI for us-east-1.
XGBOOST_IMAGE_URI = sagemaker.image_uris.retrieve(
    framework="xgboost", region=REGION, version="1.5-1"
)
print(f"XGBoost image URI: {XGBOOST_IMAGE_URI}")


def submit_training_job(job_name: str, max_depth: int, eta: float) -> None:
    sm.create_training_job(
        TrainingJobName=job_name,
        AlgorithmSpecification={
            "TrainingImage": XGBOOST_IMAGE_URI,
            "TrainingInputMode": "File",
        },
        RoleArn=TRAINING_ROLE_ARN,
        InputDataConfig=[
            {
                "ChannelName": "train",
                "DataSource": {
                    "S3DataSource": {
                        "S3DataType": "S3Prefix",
                        "S3Uri": f"s3://{BUCKET_NAME}/input/train.csv",
                        "S3DataDistributionType": "FullyReplicated",
                    }
                },
                "ContentType": "text/csv",
            },
            {
                "ChannelName": "validation",
                "DataSource": {
                    "S3DataSource": {
                        "S3DataType": "S3Prefix",
                        "S3Uri": f"s3://{BUCKET_NAME}/input/validation.csv",
                        "S3DataDistributionType": "FullyReplicated",
                    }
                },
                "ContentType": "text/csv",
            },
        ],
        OutputDataConfig={"S3OutputPath": f"s3://{BUCKET_NAME}/output/{job_name}/"},
        ResourceConfig={
            "InstanceType": "ml.m5.large",
            "InstanceCount": 1,
            "VolumeSizeInGB": 10,
        },
        StoppingCondition={"MaxRuntimeInSeconds": 900},
        HyperParameters={
            "objective": "binary:logistic",
            "num_round": "50",
            "max_depth": str(max_depth),
            "eta": str(eta),
            "eval_metric": "auc",
        },
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )


# YOUR CODE: Submit both training jobs by calling submit_training_job() twice.
# Job v1: submit_training_job(TRAINING_JOB_V1, max_depth=5, eta=0.2).
# Job v2: submit_training_job(TRAINING_JOB_V2, max_depth=8, eta=0.1).
print(f"Submitted training jobs: {TRAINING_JOB_V1}, {TRAINING_JOB_V2}")

print("Both XGBoost jobs are training, give them about 5 minutes each...")


def wait_for_training(job_name: str, ceiling: int = 900) -> str:
    elapsed = 0
    while elapsed < ceiling:
        resp = sm.describe_training_job(TrainingJobName=job_name)
        status = resp["TrainingJobStatus"]
        if status in ("Completed", "Failed", "Stopped"):
            return status
        time.sleep(15)
        elapsed += 15
        if elapsed % 60 == 0:
            print(f"  {job_name}: {elapsed}s elapsed, status: {status}")
    return "Timeout"


status_v1 = wait_for_training(TRAINING_JOB_V1)
status_v2 = wait_for_training(TRAINING_JOB_V2)
print(f"  {TRAINING_JOB_V1}: {status_v1}")
print(f"  {TRAINING_JOB_V2}: {status_v2}")

if status_v1 == "Completed":
    TRAINING_V1_ARTIFACT = sm.describe_training_job(TrainingJobName=TRAINING_JOB_V1)[
        "ModelArtifacts"
    ]["S3ModelArtifacts"]
    print(f"V1 artifact: {TRAINING_V1_ARTIFACT}")
if status_v2 == "Completed":
    TRAINING_V2_ARTIFACT = sm.describe_training_job(TrainingJobName=TRAINING_JOB_V2)[
        "ModelArtifacts"
    ]["S3ModelArtifacts"]
    print(f"V2 artifact: {TRAINING_V2_ARTIFACT}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: training and validation CSVs are in S3, both training jobs
# reached Completed, and both model.tar.gz artifacts exist.

def checkpoint_1(session):
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        for key in ("input/train.csv", "input/validation.csv"):
            try:
                s3_client.head_object(Bucket=BUCKET_NAME, Key=key)
            except ClientError as e:
                code_str = e.response["Error"]["Code"]
                if code_str in ("404", "NoSuchKey", "NoSuchBucket"):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"s3://{BUCKET_NAME}/{key} was not found. Upload the "
                            f"split files before running the checkpoint."
                        ),
                    )
                return CheckpointResult(status="error", error_reason=str(e))

        for job_name in (TRAINING_JOB_V1, TRAINING_JOB_V2):
            try:
                job = sm_client.describe_training_job(TrainingJobName=job_name)
            except ClientError as e:
                if e.response["Error"]["Code"] in ("ResourceNotFound", "ValidationException"):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"Training job {job_name} was not found. Submit both jobs "
                            f"before running the checkpoint."
                        ),
                    )
                return CheckpointResult(status="error", error_reason=str(e))
            if job["TrainingJobStatus"] != "Completed":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Training job {job_name} status is {job['TrainingJobStatus']}, "
                        f"not Completed. Wait for the poll loop in the task cell to finish."
                    ),
                )
            artifact_url = job["ModelArtifacts"]["S3ModelArtifacts"]
            # Confirm the artifact file is actually in S3.
            bucket = artifact_url.replace("s3://", "").split("/", 1)[0]
            key = artifact_url.replace("s3://", "").split("/", 1)[1]
            try:
                s3_client.head_object(Bucket=bucket, Key=key)
            except ClientError as e:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Training job {job_name} reported Completed but the model "
                        f"artifact {artifact_url} is missing from S3."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Five blanks: bucket create, two object uploads, role create, inline policy attach, and the two training-job submissions. The data generator and helper functions are already wired up. The checkpoint message tells you whether a file is missing, a job is still running, or an artifact never landed.

</details>

<details><summary>Hint 2 (direction)</summary>

In `us-east-1` use plain `s3.create_bucket(Bucket=BUCKET_NAME)` with no `CreateBucketConfiguration`. Both train and validation upload with `s3.put_object(Bucket=BUCKET_NAME, Key=..., Body=...)`. The role is created with `iam.create_role(RoleName=..., AssumeRolePolicyDocument=json.dumps(trust_policy), Tags=[...])`. The inline policy attaches with `iam.put_role_policy(RoleName=..., PolicyName=..., PolicyDocument=json.dumps(inline_policy))`. The training jobs both call the `submit_training_job()` helper already defined; v1 takes `max_depth=5, eta=0.2`, v2 takes `max_depth=8, eta=0.1`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1 (the lines you need to fill in):

```python
s3.create_bucket(Bucket=BUCKET_NAME)

s3.put_object(Bucket=BUCKET_NAME, Key="input/train.csv", Body=train_bytes)
s3.put_object(Bucket=BUCKET_NAME, Key="input/validation.csv", Body=val_bytes)

iam.create_role(
    RoleName=TRAINING_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Description="labrun mla lab 07 training and endpoint role",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

iam.put_role_policy(
    RoleName=TRAINING_ROLE_NAME,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(inline_policy),
)

submit_training_job(TRAINING_JOB_V1, max_depth=5, eta=0.2)
submit_training_job(TRAINING_JOB_V2, max_depth=8, eta=0.1)
```

</details>

**Wallet check.** Two XGBoost training jobs on ml.m5.large at $0.115 per hour for roughly 5 minutes each lands at about two pennies. The S3 control-plane plus four put_object calls sits inside the always-free tier. IAM and STS are free. Damage so far: about $0.02. Your morning coffee cost two hundred times more.

## Task 2: Create the Model Package Group and register Version 1 with PendingManualApproval

The audit trail starts with a Model Package Group. A Group is the per-model lineage container; every version of the model belongs to one Group. Inside the Group you register Model Package Versions, each pointing at a specific `model.tar.gz` and inference container.

Build it in this order:

1. Create the Group with `sm.create_model_package_group(ModelPackageGroupName=GROUP_NAME, ...)` and tag it with the lab slug.
2. Look up the XGBoost INFERENCE container URI via `sagemaker.image_uris.retrieve(framework="xgboost", region=REGION, version="1.5-1")`. The training image and the inference image have the same URI for the built-in XGBoost container, so you can reuse `XGBOOST_IMAGE_URI` from Task 1.
3. Register the v1 artifact via `sm.create_model_package` with `ModelPackageGroupName=GROUP_NAME`, `ModelApprovalStatus="PendingManualApproval"`, and an `InferenceSpecification.Containers[0]` pointing at the v1 artifact URL plus the inference container URI.

`InferenceSpecification` is the contract that tells future endpoints how to invoke the model: which container image, what input MIME types it accepts, what output MIME types it produces. For XGBoost, both are `text/csv`.

PendingManualApproval is the default state. Nothing about this state prevents another team member from constructing a SageMaker Model from this package and deploying. The exam point is that PendingManualApproval is an AUDIT CONTROL, not an ENFORCEMENT CONTROL. The reflection cell walks through the IAM Condition that turns it into an enforcement control.

In [ ]:
# NBVAL_SKIP
# Task 2: Create the Model Package Group, register Version 1 with
# PendingManualApproval.

# YOUR CODE: Create the Model Package Group using
# sm.create_model_package_group() with
# ModelPackageGroupName=GROUP_NAME,
# ModelPackageGroupDescription="labrun mla lab 07 model package group",
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Created Model Package Group: {GROUP_NAME}")

inference_spec_v1 = {
    "Containers": [
        {
            "Image": XGBOOST_IMAGE_URI,
            "ModelDataUrl": TRAINING_V1_ARTIFACT,
        }
    ],
    "SupportedContentTypes": ["text/csv"],
    "SupportedResponseMIMETypes": ["text/csv"],
    "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large", "ml.t2.medium"],
    "SupportedTransformInstanceTypes": ["ml.m5.large"],
}

# YOUR CODE: Register Model Package Version 1 using sm.create_model_package()
# with ModelPackageGroupName=GROUP_NAME,
# ModelPackageDescription="Version 1 (max_depth=5, eta=0.2)",
# InferenceSpecification=inference_spec_v1,
# ModelApprovalStatus="PendingManualApproval",
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
# Capture the returned ModelPackageArn and assign it to PACKAGE_V1_ARN.
PACKAGE_V1_ARN = None  # replace with the create_model_package call
print(f"Registered Model Package Version 1: {PACKAGE_V1_ARN}")
print("ModelApprovalStatus: PendingManualApproval")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Model Package Group exists and Model Package Version 1 is
# registered with PendingManualApproval, pointing at the v1 artifact and the
# XGBoost inference container.

def checkpoint_2(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            group = sm_client.describe_model_package_group(ModelPackageGroupName=GROUP_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Model Package Group {GROUP_NAME} was not found. Create the "
                        f"group before registering the package."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        packages = sm_client.list_model_packages(ModelPackageGroupName=GROUP_NAME).get(
            "ModelPackageSummaryList", []
        )
        if not packages:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Model Package Group {GROUP_NAME} has no registered versions yet. "
                    f"Call create_model_package() with ModelPackageGroupName={GROUP_NAME}."
                ),
            )

        if PACKAGE_V1_ARN is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"PACKAGE_V1_ARN is None. Capture the returned ModelPackageArn from "
                    f"sm.create_model_package() and assign it before running the checkpoint."
                ),
            )

        pkg = sm_client.describe_model_package(ModelPackageName=PACKAGE_V1_ARN)
        if pkg.get("ModelApprovalStatus") != "PendingManualApproval":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Model Package Version 1 ModelApprovalStatus is "
                    f"{pkg.get('ModelApprovalStatus')!r}, expected PendingManualApproval. "
                    f"Re-create the package with ModelApprovalStatus='PendingManualApproval'."
                ),
            )
        containers = pkg.get("InferenceSpecification", {}).get("Containers", [])
        if not containers:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Model Package Version 1 has no InferenceSpecification.Containers. "
                    f"Set InferenceSpecification.Containers[0].Image and ModelDataUrl in "
                    f"create_model_package()."
                ),
            )
        model_data = containers[0].get("ModelDataUrl", "")
        if TRAINING_V1_ARTIFACT and model_data != TRAINING_V1_ARTIFACT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Model Package Version 1 points at {model_data!r}, expected "
                    f"{TRAINING_V1_ARTIFACT!r}. Re-register the package with the v1 artifact."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Two blanks: create the Group, then create the Model Package Version 1 inside it. The `InferenceSpecification` is already built; you just pass it through. The checkpoint message tells you whether the Group is missing, the package never registered, or the status is wrong.

</details>

<details><summary>Hint 2 (direction)</summary>

`sm.create_model_package_group(ModelPackageGroupName=..., ModelPackageGroupDescription=..., Tags=[...])` returns immediately. `sm.create_model_package(ModelPackageGroupName=..., InferenceSpecification=inference_spec_v1, ModelApprovalStatus="PendingManualApproval", Tags=[...])` returns a dict whose `ModelPackageArn` field is the v1 ARN. Capture it into `PACKAGE_V1_ARN` so the next checkpoints can find the package.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2 (the two blocks you need to fill in):

```python
sm.create_model_package_group(
    ModelPackageGroupName=GROUP_NAME,
    ModelPackageGroupDescription="labrun mla lab 07 model package group",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

v1_response = sm.create_model_package(
    ModelPackageGroupName=GROUP_NAME,
    ModelPackageDescription="Version 1 (max_depth=5, eta=0.2)",
    InferenceSpecification=inference_spec_v1,
    ModelApprovalStatus="PendingManualApproval",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
PACKAGE_V1_ARN = v1_response["ModelPackageArn"]
```

</details>

**Wallet check.** Model Registry is free. Both `create_model_package_group` and `create_model_package` are metadata-only calls; no compute, no storage charge beyond the model.tar.gz already in S3. Damage so far: still about $0.02. Your coffee continues to outspend this lab.

## Task 3: Transition Model Package Version 1 from PendingManualApproval to Approved

There is no `approve_model_package` API. The only path to change `ModelApprovalStatus` is `update_model_package`. The call returns immediately; the registry records the transition with a timestamp visible in `LastModifiedTime`.

Build it in this order:

1. Call `sm.update_model_package(ModelPackageArn=PACKAGE_V1_ARN, ModelApprovalStatus="Approved")`.
2. Optionally pass `ApprovalDescription="<reason>"` to record human-readable approval context (the lab uses "QA passed; deploying to staging").
3. Confirm via `sm.describe_model_package(ModelPackageName=PACKAGE_V1_ARN)` that the status now reads `Approved`.

Approval status transitions are the audit trail. A regulator asking "who approved this model" wants the `ApprovalDescription` plus the `LastModifiedTime` plus the IAM caller that made the call (visible in CloudTrail). The package can transition back: `update_model_package(ModelApprovalStatus="Rejected")` reverses an approval, which is what Task 5 does for v2.

In [ ]:
# NBVAL_SKIP
# Task 3: Update v1 from PendingManualApproval to Approved.

# YOUR CODE: Update Model Package Version 1 using sm.update_model_package()
# with ModelPackageArn=PACKAGE_V1_ARN, ModelApprovalStatus="Approved",
# ApprovalDescription="QA passed; deploying to staging".
print(f"Transitioned {PACKAGE_V1_ARN} to Approved.")

# Confirm the transition landed.
post = sm.describe_model_package(ModelPackageName=PACKAGE_V1_ARN)
print(f"Current ModelApprovalStatus: {post.get('ModelApprovalStatus')}")
print(f"ApprovalDescription: {post.get('ApprovalDescription')}")
print(f"LastModifiedTime: {post.get('LastModifiedTime')}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Model Package Version 1 ModelApprovalStatus is Approved.

def checkpoint_3(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        if PACKAGE_V1_ARN is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"PACKAGE_V1_ARN is None. Task 2 must complete before Task 3."
                ),
            )

        pkg = sm_client.describe_model_package(ModelPackageName=PACKAGE_V1_ARN)
        if pkg.get("ModelApprovalStatus") != "Approved":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Model Package Version 1 ModelApprovalStatus is "
                    f"{pkg.get('ModelApprovalStatus')!r}, expected Approved. Call "
                    f"sm.update_model_package(ModelPackageArn=PACKAGE_V1_ARN, "
                    f"ModelApprovalStatus='Approved')."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

One blank: the `update_model_package` call. The checkpoint message tells you whether the status is still `PendingManualApproval` (the update did not fire) or set to something else by mistake.

</details>

<details><summary>Hint 2 (direction)</summary>

`sm.update_model_package(ModelPackageArn=..., ModelApprovalStatus="Approved", ApprovalDescription="...")` returns the updated package summary. The `ApprovalDescription` is optional but it is the only place the registry records human approval context. Capture it for the audit trail.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3 (the one block you need to fill in):

```python
sm.update_model_package(
    ModelPackageArn=PACKAGE_V1_ARN,
    ModelApprovalStatus="Approved",
    ApprovalDescription="QA passed; deploying to staging",
)
```

</details>

**Wallet check.** Another metadata-only registry call. Free. Damage so far: still about $0.02. The audit team thanks you for the timestamp.

## Task 4: Deploy the Approved Version 1 to a Serverless Inference endpoint and run 10 test predictions

The approved version is what production sees. Deploy it to a Serverless Inference endpoint, which is $0 idle plus per-invocation. Build the SageMaker Model from the Model Package ARN, build the endpoint config with a `ServerlessConfig` block, create the endpoint, wait for `InService`, then invoke 10 times against the test data.

Build it in this order:

1. Create the SageMaker Model from the Model Package ARN via `sm.create_model(ModelName=MODEL_NAME, ExecutionRoleArn=TRAINING_ROLE_ARN, Containers=[{"ModelPackageName": PACKAGE_V1_ARN}])`. The container spec resolves to the same `InferenceSpecification` you registered in Task 2.
2. Create the endpoint config via `sm.create_endpoint_config` with a `ProductionVariants[0].ServerlessConfig` block of `MemorySizeInMB=2048, MaxConcurrency=20`.
3. Create the endpoint via `sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_CONFIG_NAME)`. Poll `describe_endpoint` until `EndpointStatus` is `InService` (typically 60-90 seconds).
4. Invoke the endpoint 10 times. Each invocation sends a CSV row of 12 features and parses the returned probability.

Serverless cold-start latency is the trade-off. The first invocation after deployment is 2-5 seconds; subsequent invocations are 100-300 ms. The lab does not care about the latency number here (Lab 08 makes that the explicit comparison); the point in this lab is that the deployment came from a versioned, audited Model Package ARN.

In [ ]:
# NBVAL_SKIP
# Task 4: Deploy Approved v1 to Serverless Inference, invoke 10 times.

sm_runtime = boto3.client(
    "sagemaker-runtime",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the SageMaker Model using sm.create_model() with
# ModelName=MODEL_NAME, ExecutionRoleArn=TRAINING_ROLE_ARN,
# Containers=[{"ModelPackageName": PACKAGE_V1_ARN}],
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Created SageMaker Model: {MODEL_NAME}")

# YOUR CODE: Create the endpoint config using sm.create_endpoint_config() with
# EndpointConfigName=ENDPOINT_CONFIG_NAME,
# ProductionVariants=[{
#     "VariantName": "AllTraffic",
#     "ModelName": MODEL_NAME,
#     "ServerlessConfig": {"MemorySizeInMB": 2048, "MaxConcurrency": 20},
# }],
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Created endpoint config: {ENDPOINT_CONFIG_NAME}")

# YOUR CODE: Create the endpoint using sm.create_endpoint() with
# EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_CONFIG_NAME,
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Creating endpoint: {ENDPOINT_NAME}")

print("Serverless endpoint is warming up, give it about a minute...")
elapsed = 0
while elapsed < 600:
    resp = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    status = resp["EndpointStatus"]
    if status in ("InService", "Failed"):
        break
    time.sleep(15)
    elapsed += 15
    if elapsed % 60 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "InService":
    print(f"Endpoint reached status {status}, not InService. Check the console.")
    raise SystemExit(1)
print(f"Endpoint {ENDPOINT_NAME} is InService.")

# Invoke 10 times. Each row is 12 comma-separated features.
print("Tapping the endpoint with 10 test rows...")
predictions = []
test_rows = [
    ",".join(["0.1"] * 12),
    ",".join(["-0.5"] * 12),
    ",".join(["0.8"] * 12),
    ",".join(["-0.2"] * 12),
    ",".join(["0.3"] * 12),
    ",".join(["0.0"] * 12),
    ",".join(["1.2"] * 12),
    ",".join(["-1.0"] * 12),
    ",".join(["0.4"] * 12),
    ",".join(["0.7"] * 12),
]
for i, row in enumerate(test_rows):
    resp = sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="text/csv",
        Body=row.encode("utf-8"),
    )
    body = resp["Body"].read().decode("utf-8").strip()
    predictions.append(body)
    if i < 3:
        print(f"  row {i}: {body}")

print(f"Total predictions: {len(predictions)}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Serverless endpoint is InService and responds to 10 invocations.

def checkpoint_4(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        sm_rt = boto3.client(
            "sagemaker-runtime",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            ep = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Endpoint {ENDPOINT_NAME} was not found. Create the endpoint "
                        f"before running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if ep.get("EndpointStatus") != "InService":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Endpoint {ENDPOINT_NAME} status is "
                    f"{ep.get('EndpointStatus')!r}, not InService. Wait for the warm-up "
                    f"poll loop to finish."
                ),
            )

        config_name = ep.get("EndpointConfigName")
        cfg = sm_client.describe_endpoint_config(EndpointConfigName=config_name)
        variants = cfg.get("ProductionVariants", [])
        if not variants or "ServerlessConfig" not in variants[0]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Endpoint config {config_name} ProductionVariants[0] is not a "
                    f"Serverless config. Recreate the endpoint config with a "
                    f"ServerlessConfig block."
                ),
            )

        # Run 5 sanity invocations.
        for i in range(5):
            sample = ",".join(["0.1"] * 12)
            try:
                resp = sm_rt.invoke_endpoint(
                    EndpointName=ENDPOINT_NAME,
                    ContentType="text/csv",
                    Body=sample.encode("utf-8"),
                )
                resp["Body"].read()
            except ClientError as e:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"InvokeEndpoint failed on attempt {i + 1}: {e}. The endpoint "
                        f"is InService but not accepting predictions. Check the model's "
                        f"inference container."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Three blanks: `create_model`, `create_endpoint_config`, `create_endpoint`. The invocation loop is already written. The checkpoint message tells you whether the endpoint exists, whether it is InService, whether the config is Serverless, or whether invocations are failing.

</details>

<details><summary>Hint 2 (direction)</summary>

`sm.create_model(ModelName=..., ExecutionRoleArn=..., Containers=[{"ModelPackageName": PACKAGE_V1_ARN}], Tags=[...])` derives the inference container automatically from the Model Package's `InferenceSpecification`. The endpoint config's `ProductionVariants[0]` needs a `ServerlessConfig` dict, not an `InstanceType` (that is for real-time endpoints). `MemorySizeInMB=2048` is enough for XGBoost. `MaxConcurrency=20` is generous.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4 (the three blocks you need to fill in):

```python
sm.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=TRAINING_ROLE_ARN,
    Containers=[{"ModelPackageName": PACKAGE_V1_ARN}],
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": MODEL_NAME,
            "ServerlessConfig": {"MemorySizeInMB": 2048, "MaxConcurrency": 20},
        }
    ],
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

sm.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** Serverless Inference at $0.20 per 1 million requests plus per-GB-second. Ten invocations on a 2048 MB endpoint at roughly 200 ms each comes out to a fraction of a penny. Damage so far: still about $0.02. The endpoint will keep billing nothing while it sits idle waiting for the next request.

## Task 5: Register Version 2, Reject it, and prove the rejected version never reaches an endpoint

The approval gate is what an audit lead actually cares about. Register v2 with `PendingManualApproval`, then immediately transition to `Rejected`. Then prove via API that the deployed endpoint's underlying Model is built from the v1 artifact and NOT the v2 artifact.

Build it in this order:

1. Register Model Package Version 2 inside the same Group via `sm.create_model_package` pointing at the v2 artifact URL. Status: `PendingManualApproval`.
2. Transition v2 to `Rejected` via `sm.update_model_package(ModelPackageArn=PACKAGE_V2_ARN, ModelApprovalStatus="Rejected", ApprovalDescription="<reason>")`.
3. List endpoints tagged with this lab via `list_endpoints` plus `list_tags`. There should be exactly one.
4. Walk from the endpoint to its config, from the config to its underlying Model, from the Model to its container's `ModelDataUrl`. Confirm the URL matches the v1 artifact URL and NOT the v2 artifact URL.

The audit point: the gate works because the deployed Model's lineage is verifiable from the endpoint backwards. A rogue actor with `sagemaker:CreateModel` could technically build a Model from the Rejected v2 package and deploy it (PendingManualApproval and Rejected are not enforcement controls by default), but the platform team would notice on the next audit query because the endpoint's underlying ModelDataUrl would not match an Approved package.

In [ ]:
# NBVAL_SKIP
# Task 5: Register v2, reject it, prove the deployed endpoint still uses v1.

inference_spec_v2 = {
    "Containers": [
        {
            "Image": XGBOOST_IMAGE_URI,
            "ModelDataUrl": TRAINING_V2_ARTIFACT,
        }
    ],
    "SupportedContentTypes": ["text/csv"],
    "SupportedResponseMIMETypes": ["text/csv"],
    "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large", "ml.t2.medium"],
    "SupportedTransformInstanceTypes": ["ml.m5.large"],
}

# YOUR CODE: Register Model Package Version 2 using sm.create_model_package()
# with ModelPackageGroupName=GROUP_NAME,
# ModelPackageDescription="Version 2 (max_depth=8, eta=0.1)",
# InferenceSpecification=inference_spec_v2,
# ModelApprovalStatus="PendingManualApproval",
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
# Capture the returned ModelPackageArn into PACKAGE_V2_ARN.
PACKAGE_V2_ARN = None  # replace with the create_model_package call
print(f"Registered Model Package Version 2: {PACKAGE_V2_ARN}")

# YOUR CODE: Reject Version 2 using sm.update_model_package() with
# ModelPackageArn=PACKAGE_V2_ARN, ModelApprovalStatus="Rejected",
# ApprovalDescription="Worse validation AUC; superseded by v1".
print(f"Transitioned {PACKAGE_V2_ARN} to Rejected.")

# Walk endpoint -> config -> model -> ModelDataUrl to prove v1 not v2 is deployed.
ep = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
cfg = sm.describe_endpoint_config(EndpointConfigName=ep["EndpointConfigName"])
model_in_use = cfg["ProductionVariants"][0]["ModelName"]
model_resp = sm.describe_model(ModelName=model_in_use)
# When a Model is built from a ModelPackage, the resolved container info
# surfaces in the PrimaryContainer.ModelPackageName field.
deployed_package = model_resp.get("PrimaryContainer", {}).get("ModelPackageName")
print(f"Endpoint's underlying Model: {model_in_use}")
print(f"Model's source Model Package: {deployed_package}")
print(f"PACKAGE_V1_ARN: {PACKAGE_V1_ARN}")
print(f"PACKAGE_V2_ARN: {PACKAGE_V2_ARN}")
if deployed_package == PACKAGE_V1_ARN:
    print("Audit confirms: endpoint serves v1 (Approved). v2 (Rejected) is not deployed.")
elif deployed_package == PACKAGE_V2_ARN:
    print("AUDIT FAILURE: endpoint is serving v2 (Rejected). Investigate.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: Model Package Version 2 is Rejected and the deployed endpoint's
# underlying Model is built from the v1 package, NOT the v2 package.

def checkpoint_5(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        if PACKAGE_V2_ARN is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"PACKAGE_V2_ARN is None. Register Model Package Version 2 before "
                    f"running the checkpoint."
                ),
            )

        pkg_v2 = sm_client.describe_model_package(ModelPackageName=PACKAGE_V2_ARN)
        if pkg_v2.get("ModelApprovalStatus") != "Rejected":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Model Package Version 2 ModelApprovalStatus is "
                    f"{pkg_v2.get('ModelApprovalStatus')!r}, expected Rejected. Call "
                    f"sm.update_model_package(ModelPackageArn=PACKAGE_V2_ARN, "
                    f"ModelApprovalStatus='Rejected')."
                ),
            )

        ep = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
        cfg = sm_client.describe_endpoint_config(EndpointConfigName=ep["EndpointConfigName"])
        model_in_use = cfg["ProductionVariants"][0]["ModelName"]
        model_resp = sm_client.describe_model(ModelName=model_in_use)
        deployed_package = model_resp.get("PrimaryContainer", {}).get("ModelPackageName")
        if deployed_package == PACKAGE_V2_ARN:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Endpoint {ENDPOINT_NAME} is serving Model Package Version 2 "
                    f"(Rejected). The audit gate was bypassed. Recreate the Model "
                    f"and endpoint from PACKAGE_V1_ARN."
                ),
            )
        if deployed_package != PACKAGE_V1_ARN:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Endpoint {ENDPOINT_NAME} is serving an unexpected Model Package: "
                    f"{deployed_package!r}. Expected PACKAGE_V1_ARN={PACKAGE_V1_ARN!r}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

Two blanks: register v2 with `create_model_package`, then reject it with `update_model_package`. The endpoint-to-model walk is already written. The checkpoint message tells you whether v2 was not registered, whether it is not yet Rejected, or whether the endpoint somehow has the v2 package underneath.

</details>

<details><summary>Hint 2 (direction)</summary>

The v2 registration mirrors v1's structure exactly. Same `ModelPackageGroupName`, same `Tags`, different `ModelPackageDescription`, and `InferenceSpecification=inference_spec_v2`. The Reject call is structurally identical to the Approve call from Task 3 with `ModelApprovalStatus="Rejected"`. Do NOT deploy v2; the checkpoint relies on the endpoint still pointing at v1.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 5 (the two blocks you need to fill in):

```python
v2_response = sm.create_model_package(
    ModelPackageGroupName=GROUP_NAME,
    ModelPackageDescription="Version 2 (max_depth=8, eta=0.1)",
    InferenceSpecification=inference_spec_v2,
    ModelApprovalStatus="PendingManualApproval",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
PACKAGE_V2_ARN = v2_response["ModelPackageArn"]

sm.update_model_package(
    ModelPackageArn=PACKAGE_V2_ARN,
    ModelApprovalStatus="Rejected",
    ApprovalDescription="Worse validation AUC; superseded by v1",
)
```

</details>

**Wallet check.** Two more registry calls. Free. The Serverless endpoint is still running but billing nothing because no requests are arriving. Damage so far: still about $0.02. The audit trail buys you peace of mind that costs less than a stick of gum.

## Cleanup

The endpoint, endpoint config, and model are SageMaker resources that labrun-checks 0.6.0 does not have adapters for yet, so the cleanup cell tears them down manually before `run_cleanup` walks the IAM role and S3 bucket. The two Model Packages and the Model Package Group come down next so the registry stays tidy across re-runs. Training jobs cannot be deleted; the cleanup defensively calls `stop_training_job` on each in case the kernel was restarted mid-training.

Re-running the cell is safe. Each manual delete is wrapped in try/except that swallows "already gone" errors.

In [ ]:
# NBVAL_SKIP
# Cleanup. Manual SageMaker teardown first (no adapters in labrun-checks 0.6.0
# for endpoint/model/package), then run_cleanup walks iam_role + s3_bucket.

sm_cleanup = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

manual_warnings = []


def _swallow_already_gone(call, *args, **kwargs):
    try:
        return call(*args, **kwargs)
    except ClientError as e:
        if e.response["Error"]["Code"] in (
            "ValidationException",
            "ResourceNotFound",
            "ResourceNotFoundException",
        ):
            return None
        raise


def _safe_delete(label, action, fallback_cmd):
    try:
        action()
        print(f"Deleted {label}.")
    except ClientError as e:
        if e.response["Error"]["Code"] in (
            "ValidationException",
            "ResourceNotFound",
            "ResourceNotFoundException",
        ):
            print(f"{label} already gone, skipping.")
        else:
            manual_warnings.append(
                f"FAILED TO DELETE: {label}. Error: {e}. Run manually: {fallback_cmd}"
            )


# Stop any in-flight training jobs first.
for job_name in (TRAINING_JOB_V1, TRAINING_JOB_V2):
    try:
        resp = sm_cleanup.describe_training_job(TrainingJobName=job_name)
        if resp["TrainingJobStatus"] == "InProgress":
            try:
                sm_cleanup.stop_training_job(TrainingJobName=job_name)
                print(f"Requested stop on in-flight training job {job_name}")
            except ClientError as e:
                if e.response["Error"]["Code"] not in (
                    "ValidationException",
                    "ResourceNotFound",
                ):
                    manual_warnings.append(
                        f"FAILED TO STOP: training_job {job_name!r}. Error: {e}."
                    )
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("ValidationException", "ResourceNotFound"):
            manual_warnings.append(
                f"FAILED TO DESCRIBE: training_job {job_name!r}. Error: {e}."
            )

# Endpoint teardown (must come before endpoint-config, which must come before model).
_safe_delete(
    f"endpoint {ENDPOINT_NAME}",
    lambda: sm_cleanup.delete_endpoint(EndpointName=ENDPOINT_NAME),
    f"aws sagemaker delete-endpoint --endpoint-name {ENDPOINT_NAME}",
)

# Wait for the endpoint to actually disappear; describe_endpoint raises
# ValidationException once it is gone.
print("Waiting for endpoint to disappear...")
elapsed = 0
endpoint_gone = False
while elapsed < 180:
    try:
        sm_cleanup.describe_endpoint(EndpointName=ENDPOINT_NAME)
        time.sleep(10)
        elapsed += 10
    except ClientError as e:
        if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
            endpoint_gone = True
            break
        raise
if not endpoint_gone:
    manual_warnings.append(
        f"Endpoint {ENDPOINT_NAME} did not disappear within 3 minutes. Run manually: "
        f"aws sagemaker delete-endpoint --endpoint-name {ENDPOINT_NAME}"
    )

_safe_delete(
    f"endpoint config {ENDPOINT_CONFIG_NAME}",
    lambda: sm_cleanup.delete_endpoint_config(EndpointConfigName=ENDPOINT_CONFIG_NAME),
    f"aws sagemaker delete-endpoint-config --endpoint-config-name {ENDPOINT_CONFIG_NAME}",
)

_safe_delete(
    f"model {MODEL_NAME}",
    lambda: sm_cleanup.delete_model(ModelName=MODEL_NAME),
    f"aws sagemaker delete-model --model-name {MODEL_NAME}",
)

# Model Package Versions first, then Group (Group delete fails while versions exist).
for arn, label in ((PACKAGE_V2_ARN, "v2"), (PACKAGE_V1_ARN, "v1")):
    if arn is None:
        continue
    _safe_delete(
        f"model package {label} ({arn})",
        lambda a=arn: sm_cleanup.delete_model_package(ModelPackageName=a),
        f"aws sagemaker delete-model-package --model-package-name {arn}",
    )

_safe_delete(
    f"model package group {GROUP_NAME}",
    lambda: sm_cleanup.delete_model_package_group(ModelPackageGroupName=GROUP_NAME),
    f"aws sagemaker delete-model-package-group --model-package-group-name {GROUP_NAME}",
)

# Now hand off to run_cleanup for the IAM role and S3 bucket.
result = run_cleanup(CLEANUP_MANIFEST)

for warning in manual_warnings:
    print(warning)
for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if manual_warnings or result.warnings or result.orphans:
    print()

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

# Accounting:
# critical: the Serverless endpoint (lineage-critical even though $0 idle). 1 total.
critical_total = 1
critical_destroyed = 1 if endpoint_gone else 0
standard_total = len(CLEANUP_MANIFEST)
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids) + len(manual_warnings)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed} of {critical_total}")
print(f"Standard resources destroyed: {standard_destroyed} of {standard_total}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

final_status = "clean" if (failed_count == 0 and result.status == "clean" and endpoint_gone) else "dirty"
cleanup(status=final_status)

if failed_count > 0 or not endpoint_gone:
    sys.exit(1)

**Session total: about $0.02.** Two XGBoost training jobs on ml.m5.large at $0.115 per hour for roughly 5 minutes each comes to two pennies. The Serverless endpoint's 10 invocations come to fractions of a penny. Model Registry itself is free. S3 storage for the two splits and the two model.tar.gz files is fractions of a penny. IAM and STS are free. Check your AWS Billing console in 24 hours to confirm zero ongoing charges from this lab.

## Reflection

These are not graded. They are for you.

1. Walk through the audit story SageMaker Model Registry buys you that "pass a model.tar.gz path directly to CreateEndpoint" does not. List the artifacts Model Registry adds: the Model Package Group, the Model Package Versions, the approval status timeline, the inference container reference, and the lineage from the deployed endpoint back to the registered version. Which of these would a regulator (HIPAA, SOC 2, internal audit) ask for, and how does Registry surface them? Where does the version-to-training-job lineage live, and what would you query to reconstruct it for a model that shipped six months ago?

2. The approval workflow is metadata-only by default. A user with `sagemaker:CreateModel` permission can technically construct a SageMaker Model from any Model Package regardless of approval status, including Rejected. A team that wants to ENFORCE "only Approved packages reach Models" needs an IAM policy with a Condition on `sagemaker:ModelApprovalStatus`. Walk through what that policy looks like: which actions does it Deny, on what resources, and with what Condition. Where else in the AWS-recommended MLOps pattern (CodePipeline, EventBridge, SageMaker Pipelines) does the approval status feed into?

## Exam-Style Practice

**Question 1 (Domain 2, Model Registry value proposition):**

A platform team has been deploying SageMaker endpoints by passing a `model.tar.gz` path directly to `CreateEndpointConfig`. The audit lead has asked for: (a) a versioned record of every model that ever served traffic, (b) who approved each version, and (c) a documented rollback path. Which AWS-recommended pattern delivers all three with the least operational overhead?

A. Store model.tar.gz paths in a DynamoDB table with approval status as a column; query the table at deployment time.

B. Use SageMaker Model Registry: create a Model Package Group, register each model as a Model Package Version, transition versions through PendingManualApproval to Approved, deploy from approved versions, and use list_model_packages for the audit trail.

C. Tag S3 model artifacts with approval status and use S3 Object Lock to prevent unauthorized changes.

D. Store models in CodeCommit and use the CodeCommit commit history as the audit trail.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: a custom DynamoDB table reinvents what Model Registry already provides natively. There is no operational savings versus the AWS-managed service.
- B is correct: SageMaker Model Registry is purpose-built for this exact requirement. Model Package Groups give the per-model lineage; Model Package Versions give the per-deployment history; ModelApprovalStatus transitions give the approval audit trail; deploying from a Model Package ARN gives a documented rollback (re-approve a prior version and redeploy). All three audit requirements are met with the least operational overhead.
- C is wrong: S3 tags and Object Lock are designed for storage governance, not ML lifecycle. They lack the approval-status field, the version timeline, and the per-model lineage Model Registry provides.
- D is wrong: model artifacts are binary blobs; CodeCommit stores text. The history-as-audit pattern fits source code, not model artifacts.

</details>

**Question 2 (Domain 3, deployment infrastructure selection):**

A team is deploying a fraud-scoring model. The workload has bursty traffic: spikes to 200 requests per second during business hours, near zero overnight, and unpredictable backfill bursts on weekends. The team wants to avoid paying for idle capacity overnight and on weekends. Which SageMaker endpoint type fits with the least operational overhead?

A. Real-Time endpoint on ml.c5.xlarge with auto scaling enabled.

B. Serverless Inference endpoint with `MemorySizeInMB=4096` and `MaxConcurrency=200`.

C. Async Inference endpoint with auto scaling between 0 and 5 instances.

D. Batch Transform job scheduled hourly via EventBridge.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: Real-Time endpoints bill the underlying instance continuously even at zero traffic. Auto scaling can drop to 1 instance but not 0; overnight idle cost is real.
- B is correct: Serverless Inference is $0 idle and bills per-invocation. It scales automatically up to MaxConcurrency and back to 0 with no operational overhead. The bursty traffic pattern is the canonical Serverless use case.
- C is wrong on "least operational overhead": Async endpoints CAN scale to 0 but are designed for long-running inferences (up to 60 minutes per request). For a real-time fraud-scoring use case at sub-second per request, Async is the wrong pattern.
- D is wrong: Batch Transform is not a real-time endpoint type; it processes inputs in scheduled batches. The fraud-scoring workload requires per-request inference, not batched.

</details>